# 01 — Noise Baseline

Establishes the null hypothesis before any cleaning happens.

Three claims, each with a test:

1. **The bulk is noise** — it matches Marchenko–Pastur with fitted $(q_{\text{eff}}, \sigma^2)$, shows GOE level repulsion, and has delocalised eigenvectors.
2. **The deviations are real** — a few eigenvalues exceed $\lambda_+$ and survive every null.
3. **The naive $q = N/T$ is wrong** — and we measure by how much.

Imports `src.data` and `src.spectral` only. **No estimators**: the moment you clean
something you have stopped characterising the null.

Outputs `results/tables/baseline.csv`, which every later notebook reads rather
than refitting. One noise model for the whole project.

In [ ]:
import sys, pathlib, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository root by walking up for the `src` package.  Matching on
# a hardcoded folder name is what broke before -- the check said "notebooks"
# while the directory was "notebook", so it silently put the notebook folder
# itself on sys.path and `import src` failed.  The folder has since been
# renamed, but walking up does not care what it is called, and works from the
# repo root, from this folder, or from anywhere below either.
def _find_root():
    start = globals().get("__vsc_ipynb_file__") or pathlib.Path.cwd()
    here = pathlib.Path(start).resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "__init__.py").exists():
            return cand
    raise RuntimeError(
        f"no src/__init__.py found at or above {here}; open this notebook from "
        "inside the cloned repository")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# The Jupyter kernel is frequently NOT the interpreter a terminal `pip install`
# reaches.  Name it explicitly, so a missing package points at the environment
# to fix instead of looking like the package was never installed.
_missing = [m for m in ("numpy", "pandas", "scipy", "matplotlib")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(
        f"this kernel is missing: {', '.join(_missing)}\n"
        f"  kernel interpreter : {sys.executable}\n"
        f"  install into THAT interpreter:\n"
        f'      "{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"\n'
        f"  then restart the kernel (a plain `pip install` in a terminal may\n"
        f"  have targeted a different environment entirely)")

from src.data import (load_prices, to_returns, filter_universe, prepare,
                      scramble, phase_randomise, bootstrap_iid,
                      factor_correlation, simulate_returns)
from src.spectral import (spectrum, mp_pdf, mp_cdf, mp_edges, fit_mp_bulk,
                          unfold, wigner_surmise, spacing_test, ipr)

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7, 4),
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})

FIGDIR = ROOT / "results" / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = ROOT / "results" / "tables";  TABDIR.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(0)

# Set to False once a real price file exists. Keeping the synthetic path means
# restart-and-run-all works for anyone cloning the repo without the data.
USE_SYNTHETIC = True
PRICE_FILE   = ROOT / "data" / "processed" / "prices.parquet"
N_ASSETS, WINDOW = 300, 1000
HALFLIFE = 63

## §1 — Universe and preparation

Point-in-time membership, liquidity and staleness filters, then devolatilisation.
The attrition table is the first thing a skeptical reader checks, so it is printed
rather than buried.

In [ ]:
if USE_SYNTHETIC:
    N0, T0, K = 320, 1200, 3
    C_TRUE = factor_correlation(N0, k=K, loading_sd=0.5, rng=RNG)
    R = pd.DataFrame(simulate_returns(C_TRUE, T0, rng=RNG) * 0.012,
                     index=pd.bdate_range("2019-01-02", periods=T0),
                     columns=[f"SYN{i:03d}" for i in range(N0)])
    vol = pd.Series(np.exp(np.cumsum(RNG.normal(0, 0.03, T0))), index=R.index)
    R = R.mul(vol, axis=0)                       # volatility clustering to remove
    R.iloc[:, :6] = R.iloc[:, :6].where(RNG.random((T0, 6)) > 0.25, 0.0)  # stale names
    n_raw = R.shape[1]
else:
    prices = load_prices(PRICE_FILE)
    R = to_returns(prices, kind="log")
    n_raw = R.shape[1]

R = R.tail(WINDOW)
R_f = filter_universe(R, n_assets=N_ASSETS, min_obs_frac=0.98,
                      max_zero_frac=0.05, max_abs_return=0.9)
panel = prepare(R_f, halflife=HALFLIFE)
T, N = panel.shape

print(pd.Series({
    "raw columns": n_raw,
    "after filters": R_f.shape[1],
    "final N": N,
    "final T": T,
    "q = N/T": round(N / T, 4),
}).to_string())

## §2 — The raw spectrum

Before fitting anything: what does the top of the spectrum look like, and is the
leading eigenvector the market mode it should be?

In [ ]:
evals, evecs, Corr = spectrum(panel.X)

top = pd.DataFrame({
    "lambda": evals[-10:][::-1],
    "trace_frac": evals[-10:][::-1] / N,
})
print(top.to_string(float_format=lambda v: f"{v:8.4f}"))

v1 = evecs[:, -1]
print(f"\nleading eigenvector: same-sign fraction = {max((v1>0).mean(), (v1<0).mean()):.3f}"
      f"   dispersion |v|/mean|v| = {np.std(np.abs(v1))/np.mean(np.abs(v1)):.3f}")
print("A near-uniform, single-signed leading vector is the market mode. "
      "If it is not, something upstream is wrong.")

## §3 — Fitting the bulk

Both parameters float. $\sigma^2 < 1$ because the spikes carry variance out of the
bulk; $q_{\text{eff}} > N/T$ because serial dependence and non-stationarity reduce
the effective sample size. Neither is a nuisance parameter — both are results.

In [ ]:
fit = fit_mp_bulk(evals, q0=N / T)

print(pd.Series({
    "q naive (N/T)":   N / T,
    "q_eff (fitted)":  fit["q_eff"],
    "q_eff / q":       fit["q_eff"] / (N / T),
    "sigma^2":         fit["sigma2"],
    "lambda_minus":    fit["lambda_minus"],
    "lambda_plus":     fit["lambda_plus"],
    "spikes":          fit["n_exclude"],
    "bulk KS":         fit["ks"],
}).to_string(float_format=lambda v: f"{v:9.4f}"))

phi = evals[-fit["n_exclude"]:].sum() / N
print(f"\nsigma^2 = {fit['sigma2']:.3f} says {100*(1-fit['sigma2']):.0f}% of the trace has "
      f"left the bulk (direct measurement: {100*phi:.0f}%).")
print(f"q_eff/q = {fit['q_eff']/(N/T):.2f} says {T} observations behave like "
      f"{int(N/fit['q_eff'])} independent ones.")

## §4 — The null comparison

**The centrepiece.** Three nulls:

| null | destroys | preserves |
|---|---|---|
| `scramble` | all cross-correlation | each column's marginal exactly, fat tails included |
| `phase_randomise` | cross-correlation | each column's autocorrelation |
| `bootstrap_iid` | time-series structure | cross-sectional correlation |

Two things must hold. Fitting the scrambled null must recover $q \approx N/T$ and
$\sigma^2 \approx 1$ — the machinery validating itself on data known to be null,
which is what licenses trusting §3. And the real spikes must have no counterpart in
any null. Because `scramble` preserves marginals exactly, it forecloses the
"it's just heavy tails" objection in a way a Gaussian simulation cannot.

In [ ]:
nulls = {
    "real":      panel.X,
    "scrambled": scramble(panel.X, RNG),
    "phase":     phase_randomise(panel.X, RNG),
    "bootstrap": bootstrap_iid(panel.X, RNG),
}

rows, spec = [], {}
for name, Y in nulls.items():
    ev, vec, _ = spectrum(Y)
    f = fit_mp_bulk(ev, q0=N / T)
    ks_sp, _ = spacing_test(ev, f["q_eff"], f["sigma2"])
    spec[name] = (ev, vec, f)
    rows.append(dict(null=name, q_eff=f["q_eff"], sigma2=f["sigma2"],
                     lambda_plus=f["lambda_plus"], spikes=f["n_exclude"],
                     bulk_ks=f["ks"], spacing_p=ks_sp.pvalue,
                     lambda_max=ev[-1]))

null_tab = pd.DataFrame(rows).set_index("null")
print(f"naive q = N/T = {N/T:.4f}\n")
print(null_tab.to_string(float_format=lambda v: f"{v:9.4f}"))

ok_q  = abs(null_tab.loc["scrambled", "q_eff"] - N / T) / (N / T) < 0.10
ok_s2 = abs(null_tab.loc["scrambled", "sigma2"] - 1.0) < 0.05
print(f"\nself-validation: scrambled recovers q [{'PASS' if ok_q else 'FAIL'}], "
      f"sigma^2 [{'PASS' if ok_s2 else 'FAIL'}]")
print(f"serial-dependence share of the q gap: "
      f"{null_tab.loc['phase','q_eff'] - null_tab.loc['scrambled','q_eff']:+.4f}")

In [ ]:
# Figure 1 — real vs nulls, with the fitted MP overlay
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bins = np.linspace(0, min(4.0, np.percentile(spec["real"][0], 99.5)), 90)

for name, style in [("real", dict(color="C3", alpha=.55)),
                    ("scrambled", dict(color="C0", alpha=.35)),
                    ("phase", dict(color="C2", alpha=.25))]:
    ax.hist(spec[name][0], bins=bins, density=True, label=name, **style)

g = np.linspace(bins[0] + 1e-6, bins[-1], 800)
f_r, f_s = spec["real"][2], spec["scrambled"][2]
ax.plot(g, mp_pdf(g, f_r["q_eff"], f_r["sigma2"]), "C3-", lw=2,
        label=f"MP fit real (q={f_r['q_eff']:.2f}, $\\sigma^2$={f_r['sigma2']:.2f})")
ax.plot(g, mp_pdf(g, f_s["q_eff"], f_s["sigma2"]), "C0--", lw=2,
        label=f"MP fit scrambled (q={f_s['q_eff']:.2f}, $\\sigma^2$={f_s['sigma2']:.2f})")
ax.axvline(f_r["lambda_plus"], color="C3", ls=":", lw=1.2)

ax.set_xlabel("eigenvalue $\\lambda$"); ax.set_ylabel("density")
ax.set_title("Empirical spectrum vs nulls, with fitted Marchenko-Pastur")
ax.legend(fontsize=7.5); fig.tight_layout()
fig.savefig(FIGDIR / "01_spectrum_vs_nulls.png", bbox_inches="tight"); plt.show()

In [ ]:
# Figure 2 — log tail: where the real spectrum leaves every null behind
fig, ax = plt.subplots(figsize=(7.5, 4))
for name, c in [("real", "C3"), ("scrambled", "C0"), ("phase", "C2"), ("bootstrap", "C1")]:
    ev = spec[name][0]
    ax.semilogy(np.arange(1, 41), ev[-40:][::-1], "o-", ms=3, color=c, label=name)
ax.axhline(f_r["lambda_plus"], color="k", ls=":", lw=1,
           label=f"$\\lambda_+$ = {f_r['lambda_plus']:.2f}")
ax.set_xlabel("rank"); ax.set_ylabel("$\\lambda$ (log)")
ax.set_title("Top 40 eigenvalues: real spikes have no counterpart in any null")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "01_tail_loglinear.png", bbox_inches="tight"); plt.show()

## §5 — Universality

Density agreement is weak evidence: MP is a smooth two-parameter unimodal density
and plenty of wrong models fit such data acceptably.

**Level spacings are strong evidence.** Random matrix eigenvalues repel —
$p(s) \to 0$ as $s \to 0$ — because coincident eigenvalues are a codimension-2 event
in the space of symmetric matrices. Independent eigenvalues would give Poisson
spacings with $p(0) = 1$. The two are qualitatively different at the origin, and no
factor model reproduces repulsion.

Unfolding is required first: spacings mean nothing until the local density is
divided out. Note the dependency — unfolding uses the §3 fit, so a bad fit produces
a spurious spacing failure.

In [ ]:
ks_real, s_real = spacing_test(evals, fit["q_eff"], fit["sigma2"])
ks_scr,  s_scr  = spacing_test(*spec["scrambled"][0:1],
                               spec["scrambled"][2]["q_eff"],
                               spec["scrambled"][2]["sigma2"]) \
                  if False else spacing_test(spec["scrambled"][0],
                                             spec["scrambled"][2]["q_eff"],
                                             spec["scrambled"][2]["sigma2"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(s_real, bins=40, density=True, alpha=.5, color="C3", label="real (unfolded)")
ax.hist(s_scr,  bins=40, density=True, alpha=.3, color="C0", label="scrambled")
u = np.linspace(0, 4, 400)
ax.plot(u, wigner_surmise(u), "k-",  lw=2, label="Wigner surmise (GOE)")
ax.plot(u, np.exp(-u),        "k--", lw=1.5, label="Poisson (independent)")
ax.set_xlabel("normalised spacing $s$"); ax.set_ylabel("$p(s)$")
ax.set_title(f"Level repulsion — KS p = {ks_real.pvalue:.3f} (real), {ks_scr.pvalue:.3f} (scrambled)")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "01_level_spacings.png", bbox_inches="tight"); plt.show()

print(f"real      spacing KS = {ks_real.statistic:.4f}, p = {ks_real.pvalue:.4f}")
print(f"scrambled spacing KS = {ks_scr.statistic:.4f}, p = {ks_scr.pvalue:.4f}")

In [ ]:
# Inverse participation ratio: physics test and data-quality alarm in one
ipr_all = ipr(evecs)
pt = 3.0 / N
bulk_mask = evals <= fit["lambda_plus"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(evals[bulk_mask], ipr_all[bulk_mask], ".", ms=4, color="C0", label="bulk")
ax.semilogy(evals[~bulk_mask], ipr_all[~bulk_mask], "o", ms=6, color="C3", label="spikes")
ax.axhline(pt, color="k", ls="--", lw=1, label=f"Porter-Thomas 3/N = {pt:.4f}")
ax.axhline(10 * pt, color="k", ls=":", lw=1, label="10x (localisation flag)")
ax.set_xlabel("$\\lambda$"); ax.set_ylabel("IPR (log)")
ax.set_title("Eigenvector delocalisation"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR / "01_ipr.png", bbox_inches="tight"); plt.show()

flag = np.where(bulk_mask & (ipr_all > 10 * pt))[0]
print(f"median bulk IPR = {np.median(ipr_all[bulk_mask]):.5f}  vs 3/N = {pt:.5f}")
if flag.size:
    print(f"\n{flag.size} localised bulk eigenvector(s) — dominant names:")
    for i in flag[:8]:
        j = int(np.argmax(np.abs(evecs[:, i])))
        print(f"  lambda={evals[i]:7.4f}  IPR={ipr_all[i]:.4f}  -> {panel.tickers[j]} "
              f"(weight {evecs[j, i]:+.3f})")
    print("Localised bulk vectors are almost always stale or illiquid series, "
          "not factors. They corrupt the lower edge, where the RIE is weakest.")
else:
    print("no localised bulk eigenvectors flagged")

## §6 — Stability

Are the headline numbers a property of the market or of one arbitrary window?
This sweep is also how $(N, T)$ gets chosen for everything downstream: we want
$q \in [0.3, 0.8]$, where there is enough noise for cleaning to matter and the
asymptotics still bite.

In [ ]:
sweep = []
for T_w in [250, 500, 750, 1000, 1500]:
    for N_w in [100, 200, 300]:
        Rw = R.tail(T_w)
        Rf = filter_universe(Rw, n_assets=N_w, min_obs_frac=0.95, max_zero_frac=0.05)
        if Rf.shape[1] < 50 or Rf.shape[0] < 120:
            continue
        p = prepare(Rf, halflife=HALFLIFE)
        Tw, Nw = p.shape
        ev_w, _, _ = spectrum(p.X)
        f_w = fit_mp_bulk(ev_w, q0=Nw / Tw)
        sweep.append(dict(T=Tw, N=Nw, q=Nw / Tw, q_eff=f_w["q_eff"],
                          gap=f_w["q_eff"] - Nw / Tw, sigma2=f_w["sigma2"],
                          spikes=f_w["n_exclude"]))

sweep = pd.DataFrame(sweep).drop_duplicates(subset=["T", "N"])
print(sweep.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for Nw, grp in sweep.groupby("N"):
    axes[0].plot(grp["T"], grp["gap"], "o-", ms=4, label=f"N={Nw}")
    axes[1].plot(grp["T"], grp["spikes"], "o-", ms=4, label=f"N={Nw}")
axes[0].set_ylabel("$q_{eff} - N/T$"); axes[1].set_ylabel("spike count")
for a in axes:
    a.set_xlabel("window length T"); a.legend(fontsize=8)
axes[0].axhline(0, color="k", lw=.8)
fig.suptitle("Stability of the noise model across windows", y=1.02)
fig.tight_layout(); fig.savefig(FIGDIR / "01_stability.png", bbox_inches="tight"); plt.show()

## §7 — Conclusions and handoff

Later notebooks read `results/tables/baseline.csv` rather than refitting, so the
whole project uses one noise model. Estimators must be configured with
`q_eff`, **not** `N/T`.

In [ ]:
baseline = pd.DataFrame([dict(
    N=N, T=T, q_naive=N / T,
    q_eff=fit["q_eff"], sigma2=fit["sigma2"],
    lambda_minus=fit["lambda_minus"], lambda_plus=fit["lambda_plus"],
    n_spikes=fit["n_exclude"], bulk_ks=fit["ks"],
    spacing_p=ks_real.pvalue,
    ipr_median_bulk=float(np.median(ipr_all[bulk_mask])), ipr_pt=3.0 / N,
    halflife=HALFLIFE, synthetic=USE_SYNTHETIC,
)])
baseline.to_csv(TABDIR / "baseline.csv", index=False)
null_tab.to_csv(TABDIR / "baseline_nulls.csv")

print(baseline.T.to_string(header=False, float_format=lambda v: f"{v:10.4f}"))

lines = [
    "",
    "CONCLUSION",
    "----------",
    f"The bulk is Marchenko-Pastur with q_eff = {fit['q_eff']:.3f} against a naive",
    f"N/T = {N/T:.3f}, and sigma^2 = {fit['sigma2']:.3f}: {100*(1-fit['sigma2']):.0f}% of the trace sits",
    f"outside the bulk. It passes the GOE spacing test at p = {ks_real.pvalue:.3f} and its",
    f"eigenvectors are delocalised (median bulk IPR {np.median(ipr_all[bulk_mask]):.5f} vs 3/N = {3/N:.5f}).",
    f"{fit['n_exclude']} eigenvalues exceed lambda_+ = {fit['lambda_plus']:.3f} and have no counterpart",
    "in any of the three nulls.",
    "",
    "The cleaning problem is therefore well posed, and all downstream estimators",
    f"should be configured with q_eff = {fit['q_eff']:.3f}, not N/T = {N/T:.3f}.",
    "",
    "Caveat: spike counting by thresholding is noisy at the edge. The largest bulk",
    "eigenvalue fluctuates on the Tracy-Widom scale N^(-2/3) around lambda_+, so one",
    "or two crossings are expected. A Tracy-Widom test is the principled replacement.",
]
print(chr(10).join(lines))
